# Progressive Student Model Training

Trains a split pipeline for distilling our Claude-based extraction pipeline into local models:
- **NER**: BioBERT-base (110M) with BIOES tagging + sliding window
- **RE**: BioBERT-base (110M) pair classifier with typed entity markers

Trained at progressive scales: 1 → 10 → 100 → 400 BioRED gold examples.

**Runtime**: Select GPU (Runtime → Change runtime type → T4 GPU)

## Setup

In [ ]:
# Mount Google Drive (optional — for persisting results across sessions)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repo and install dependencies (torch is pre-installed on Colab)
!git clone https://github.com/Currie32/biolink-hub.git /content/biolink-hub
%cd /content/biolink-hub
!pip install -e '.[train]' -q

In [ ]:
# Verify GPU
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s %(message)s",
)

## Quick smoke test (N=1, N=10)

Run the split pipeline at the two smallest scales to validate everything works.
Expected time: ~10 min on T4 (BioBERT-base is 110M params).

In [ ]:
from bioextract.model.train_progressive import run_progressive

results_small = run_progressive(scales=[1, 10])

### Smoke test checks

- **N=1**: The pipeline should memorize the single example
- **N=10**: NER should tag at least some entities on unseen text; RE should predict a few relationships

In [ ]:
import json
print(json.dumps(results_small, indent=2))

## Full experiment (N=100, N=400)

Run the larger scales. Expected time: ~2-3 hr on T4 (BioBERT-base is 110M params).

You can run these individually if you're worried about session timeouts.

In [ ]:
results_large = run_progressive(scales=[100, 400])

## Review results

In [ ]:
# Load the full results file (includes all scales run so far)
import json
from pathlib import Path

results_path = Path("models/progressive_results.json")
with open(results_path) as f:
    all_results = json.load(f)

print(json.dumps(all_results, indent=2))

In [ ]:
from bioextract.model.train_progressive import _print_summary
_print_summary(all_results)

## Try the model

Test any trained model on sample text before saving. Set `N` to whichever scale you trained.

## Save student model

Package the trained model for local use. Set `N` to the scale you want to save.

In [ ]:
import shutil
from pathlib import Path

N = 100  # change to match the scale you trained

ner_src = Path(f"models/ner_n{N}")
re_src  = Path(f"models/re_n{N}")

for p in [ner_src, re_src]:
    if not p.exists():
        raise RuntimeError(f"Model not found: {p} — check N and re-run training")

# Inference-only files (skips checkpoint-N/ dirs which inflate size by 10×)
BERT_FILES = [
    "config.json", "pytorch_model.bin", "model.safetensors",
    "tokenizer_config.json", "tokenizer.json", "vocab.txt",
    "special_tokens_map.json",
]

def copy_inference_files(src: Path, dest: Path, extra: list[str] = []):
    dest.mkdir(parents=True, exist_ok=True)
    for fname in BERT_FILES + extra:
        f = src / fname
        if f.exists():
            shutil.copy(f, dest / fname)
    print(f"  {dest.name}/: {[f for f in BERT_FILES + extra if (src / f).exists()]}")

stage = Path("/content/student_model_stage")
if stage.exists():
    shutil.rmtree(stage)

copy_inference_files(ner_src, stage / "ner")
copy_inference_files(re_src,  stage / "re", extra=["re_heads.pt"])

# Zip for download
zip_path = Path("/content/student_model.zip")
shutil.make_archive(str(zip_path.with_suffix("")), "zip", stage)
print(f"\nZip: {zip_path} ({zip_path.stat().st_size / 1e6:.0f} MB)")

# Backup to Drive
drive_dest = Path("/content/drive/MyDrive/biolink-hub/student_model")
if drive_dest.exists():
    shutil.rmtree(drive_dest)
shutil.copytree(stage, drive_dest)
print(f"Saved to Drive: {drive_dest}")

In [ ]:
# Download the zip to your local machine
from google.colab import files
files.download("/content/student_model.zip")